# FlashEats — Class 5 Investigation
**Mission:** assemble trustworthy evidence about late deliveries.

Rules: no ML for first 90 minutes; preserve raw API responses; state your definition of late; never silently drop failures.

> **Approach for this starter:** answer every challenge **SQL-first**, inside the database, as the starter's query cells intend.
> The full pandas investigation of the same questions is in `FlashEats_Class5_Student.ipynb`.

In [1]:
import os, json, sqlite3
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 200)


def find_pack_root():
    """Pack root = folder with database/flasheats.db AND the Class 6/7 files (the updated instructor pack).

    Search order: $FLASHEATS_PACK, Challenges/flasheats-classroom-pack (created by get_pack.sh),
    the working directory and its parents, ~/flasheats-classroom-pack, then the Colab location.
    """
    candidates = [Path(os.environ["FLASHEATS_PACK"])] if os.environ.get("FLASHEATS_PACK") else []
    candidates += [Path.cwd() / "flasheats-classroom-pack", Path.cwd() / "Challenges" / "flasheats-classroom-pack",
                   Path.cwd(), *Path.cwd().parents, Path.home() / "flasheats-classroom-pack",
                   Path("/content/FlashEats_Classroom_Pack")]
    for folder in candidates:
        if (folder / "database" / "flasheats.db").exists() and (folder / "data" / "order_outcomes.csv").exists():
            return folder
    raise FileNotFoundError("FlashEats pack not found. Run `bash Challenges/get_pack.sh` or set FLASHEATS_PACK.")


BASE = find_pack_root()
OUTPUT = Path.cwd() / "output" if (Path.cwd() / "get_pack.sh").exists() else Path.cwd() / "Challenges" / "output"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Pack:", os.path.relpath(BASE))

import sys, time, subprocess
import requests
con = sqlite3.connect(BASE / "database" / "flasheats.db")

Pack: flasheats-classroom-pack


## Challenge 1 — How large is the late-delivery problem?
Write down your definition of late, denominator, cancellation policy, and missing-timestamp policy before querying.

**Definitions (fixed before querying)**
- **Late:** `actual_delivery_at > promised_eta` (any delay above 0 min).
- **Denominator:** unique delivered orders (status compared case-insensitively) that have both timestamps.
- **Cancellations:** excluded from the rate, counted separately.
- **Missing timestamps:** excluded from the rate, **counted and reported** as unknown.
- **Grain:** one row per `order_id`, enforced with `ROW_NUMBER()`, because the table does contain duplicate IDs.

In [2]:
con=sqlite3.connect(BASE/'database'/'flasheats.db')
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'",con)

,name
0,customers
1,drivers
2,restaurants
3,orders


In [3]:
query = """
WITH one_row_per_order AS (          -- the table has 3 duplicated order_ids: keep one row each
    SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY rowid) AS rn
    FROM orders
),
o AS (
    SELECT *, LOWER(TRIM(final_status)) AS status,            -- 'Delivered' == 'delivered'
           (julianday(actual_delivery_at) - julianday(promised_eta)) * 1440 AS delay_min
    FROM one_row_per_order WHERE rn = 1
)
SELECT
    COUNT(*)                                                              AS unique_orders,
    SUM(status = 'cancelled')                                             AS cancelled_excluded,
    SUM(status = 'delivered' AND actual_delivery_at IS NULL)              AS delivered_missing_time,
    SUM(status = 'delivered' AND actual_delivery_at IS NOT NULL)          AS valid_delivered,
    SUM(status = 'delivered' AND delay_min > 0)                           AS late,
    ROUND(100.0 * SUM(status = 'delivered' AND delay_min > 0)
          / SUM(status = 'delivered' AND actual_delivery_at IS NOT NULL), 1) AS late_rate_pct,
    ROUND(100.0 * SUM(status = 'delivered' AND delay_min > 10)
          / SUM(status = 'delivered' AND actual_delivery_at IS NOT NULL), 1) AS late_over_10min_pct
FROM o;
"""
pd.read_sql(query, con)

,unique_orders,cancelled_excluded,delivered_missing_time,valid_delivered,late,late_rate_pct,late_over_10min_pct
0,1600,68,37,1495,843,56.4,23.3


In [4]:
# Data-quality checks that change the answer
pd.read_sql("""
SELECT 'duplicate order_id rows'                  AS check_name, COUNT(*) - COUNT(DISTINCT order_id) AS n FROM orders
UNION ALL SELECT 'status spelled differently',    SUM(final_status <> LOWER(final_status)) FROM orders
UNION ALL SELECT 'promised ETA before order created', SUM(julianday(promised_eta) < julianday(created_at)) FROM orders
UNION ALL SELECT 'delivered before picked up',    SUM(julianday(actual_delivery_at) < julianday(pickup_at)) FROM orders
UNION ALL SELECT 'missing restaurant_id or driver_id', SUM(restaurant_id IS NULL OR driver_id IS NULL) FROM orders
""", con)

,check_name,n
0,duplicate order_id rows,3
1,status spelled differently,5
2,promised ETA before order created,4
3,delivered before picked up,5
4,missing restaurant_id or driver_id,6


**Answer:**
- **843 of 1,495** valid delivered orders are late: **56.4%**. Only **23.3%** are more than 10 minutes late.
- **68** cancelled orders are excluded. **37** delivered orders have no delivery time; they move the rate between 55.0% and 57.4%.
- A team using `final_status = 'delivered'` with no de-duplication gets 841 / 1,493 (56.3%). That query double-counts 3 duplicate rows and misses 5 `"Delivered"` rows.
- The 4 orders whose promised ETA is *before* creation are the 4 most "late" orders. They are data errors, not operational failures.

### Checkpoint
Compare counts with another team. If they differ, investigate row grain, duplicates, cancellations, missing timestamps, and definitions.

## Challenge 2 — Operations says traffic is the cause
Test the claim. Treat traffic/weather/distance as descriptive signals, not causal proof.

In [5]:
query_traffic = """
WITH o AS (
    SELECT *, LOWER(TRIM(traffic_bucket)) AS traffic,
           (julianday(actual_delivery_at) - julianday(promised_eta)) * 1440 AS delay_min,
           (julianday(pickup_at) - julianday(created_at)) * 1440            AS order_to_pickup_min,
           (julianday(actual_delivery_at) - julianday(pickup_at)) * 1440    AS pickup_to_door_min,
           (julianday(promised_eta) - julianday(created_at)) * 1440         AS promised_min
    FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY rowid) AS rn FROM orders)
    WHERE rn = 1 AND LOWER(TRIM(final_status)) = 'delivered' AND actual_delivery_at IS NOT NULL
      AND julianday(promised_eta) >= julianday(created_at) AND julianday(actual_delivery_at) >= julianday(pickup_at)
)
SELECT {dim} AS bucket, COUNT(*) AS orders,
       ROUND(AVG(delay_min > 0), 3) AS late_rate,
       ROUND(AVG(promised_min), 1) AS avg_promised_min,
       ROUND(AVG(order_to_pickup_min), 1) AS avg_order_to_pickup_min,
       ROUND(AVG(pickup_to_door_min), 1) AS avg_pickup_to_door_min
FROM o GROUP BY 1 ORDER BY late_rate
"""
for dim in ["traffic", "weather_bucket", "CASE WHEN delay_min > 0 THEN 'late' ELSE 'on time' END"]:
    display(pd.read_sql(query_traffic.format(dim=dim), con))

,bucket,orders,late_rate,avg_promised_min,avg_order_to_pickup_min,avg_pickup_to_door_min
0,low,358,0.494,66.4,27.2,40.1
1,medium,585,0.511,68.9,27.0,43.6
2,severe,122,0.664,79.9,27.4,58.9
3,high,421,0.665,73.1,28.4,49.8


,bucket,orders,late_rate,avg_promised_min,avg_order_to_pickup_min,avg_pickup_to_door_min
0,clear,1185,0.532,69.4,27.4,44.1
1,rain,229,0.668,73.4,28.5,50.9
2,heavy_rain,72,0.736,76.7,26.7,57.7


,bucket,orders,late_rate,avg_promised_min,avg_order_to_pickup_min,avg_pickup_to_door_min
0,on time,649,0.0,69.3,19.5,43.4
1,late,837,1.0,71.2,33.7,47.7


**Answer:** traffic is **associated** with lateness, but it is not the main place delay accumulates.
- **Traffic:** severe/high traffic orders are late about 66% of the time vs 49% in low traffic. But the promise already stretches with traffic (the average promise is up to about 13 min longer).
- **Weather:** rain and heavy rain are late 67% and 74% of the time vs 53% when clear.
- **The stage split is the stronger signal:** late orders take on average about **14 more minutes from order to pickup** (33.7 vs 19.5 min), while the road stage adds only about 4 minutes (47.7 vs 43.4).

These are descriptive signals, not causal proof. Busy rainy evenings stack factors, and the order-to-pickup stage mixes restaurant preparation with driver arrival.

## Challenge 3 — Customer Support disagrees
Inspect support tickets. What are customers actually complaining about? Are ticket records unique?

In [6]:
tickets = pd.read_csv(BASE / "data" / "support_tickets.csv")
print("rows:", len(tickets), "| unique ticket_id:", tickets["ticket_id"].nunique(),
      "| exact duplicates:", int(tickets.duplicated().sum()), "| no order_id:", int(tickets["order_id"].isna().sum()))
tickets = tickets.drop_duplicates()
tickets["category_cleaned"] = tickets["category"].str.strip().str.lower().str.replace(" ", "_")
# Join the CSV to the database in SQL WITHOUT writing into the source database:
# an in-memory database holds the tickets, and flasheats.db is attached alongside it.
mem = sqlite3.connect(":memory:")
mem.execute("ATTACH DATABASE ? AS fe", (str(BASE / "database" / "flasheats.db"),))
tickets.to_sql("t", mem, index=False)
pd.read_sql("""
SELECT t.category_cleaned AS category, COUNT(*) AS tickets,
       SUM(julianday(o.actual_delivery_at) > julianday(o.promised_eta)) AS on_late_orders,
       SUM(julianday(o.actual_delivery_at) <= julianday(o.promised_eta)) AS on_on_time_orders
FROM t
LEFT JOIN (SELECT * FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY rowid) rn FROM fe.orders) WHERE rn = 1) AS o
       ON o.order_id = t.order_id
GROUP BY 1 ORDER BY tickets DESC
""", mem)

rows: 202 | unique ticket_id: 201 | exact duplicates: 1 | no order_id: 3


,category,tickets,on_late_orders,on_on_time_orders
0,late_delivery,39,33,5
1,restaurant_delay,37,28,9
2,eta_changed,37,28,7
3,ready_but_waiting,33,30,3
4,status_mismatch,29,26,2
5,driver_not_moving,25,17,8
6,eta_issue,1,1,0


**Answer:**
- **201 unique tickets** out of 202 rows: T00013 is an exact duplicate. **3 tickets have no `order_id`.**
- **Complaints are mostly about the pickup hand-off and ETA instability:** late delivery, *ETA changed*, *restaurant delay*, *ready but waiting*, *status mismatch*, *driver not moving*.
- **34 tickets are on orders that were on time**, so customers feel ETA and status problems that the late-rate KPI does not capture.
- `"ETA issue"` is kept separate from `eta_changed` until Support confirms they mean the same thing.

## Challenge 4 — Retrieve Dispatch data reliably

In [7]:
try:
    requests.get("http://127.0.0.1:8000/health", timeout=1); api_proc = None
except requests.ConnectionError:
    api_proc = subprocess.Popen([sys.executable, str(BASE / "api" / "mock_dispatch_api.py")],
                                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(2)
requests.get("http://127.0.0.1:8000/health").json()

{'service': 'flasheats-dispatch-api', 'status': 'ok'}

In [8]:
url='http://127.0.0.1:8000/dispatch/orders'
r=requests.get(url,params={'page':1,'page_size':50},timeout=10)
print(r.status_code); payload=r.json(); print(payload.keys(), len(payload.get('data',[])), payload.get('has_more'))

200
dict_keys(['data', 'has_more', 'page', 'page_size', 'total_records']) 50 True


### Your task
Implement pagination + retry + raw-page preservation. Refuse to claim success if ingestion is incomplete.

In [9]:
RAW_DIR = OUTPUT / "class5_starter_raw_dispatch"; RAW_DIR.mkdir(parents=True, exist_ok=True)


def fetch_all_dispatch_orders(page_size=50, max_attempts=4):
    records, page, retries = [], 1, []
    while True:
        for attempt in range(1, max_attempts + 1):
            r = requests.get(url, params={"page": page, "page_size": page_size}, timeout=10)
            if r.status_code == 200:
                break
            if r.status_code not in (429, 500, 502, 503, 504):
                raise RuntimeError(f"page {page}: HTTP {r.status_code} (not retryable)")
            retries.append((page, r.status_code))
            time.sleep(float(r.json().get("retry_after_seconds", 1)) if r.status_code == 429 else 2 ** (attempt - 1))
        else:
            raise RuntimeError(f"page {page}: failed after {max_attempts} attempts; ingestion INCOMPLETE")
        payload = r.json()
        (RAW_DIR / f"page_{page:03d}.json").write_text(json.dumps(payload))     # raw page preserved
        records.extend(payload["data"])
        if not payload["has_more"]:
            break
        page += 1
    total = payload["total_records"]
    unique = len({x["order_id"] for x in records})
    if len(records) != total or unique != total:
        raise RuntimeError(f"INCOMPLETE: retrieved {len(records)} ({unique} unique) of {total}")
    print(f"COMPLETE: {page} pages, {len(records)} records = API total {total}; retries: {retries}")
    return records


records = fetch_all_dispatch_orders()

COMPLETE: 32 pages, 1600 records = API total 1600; retries: [(3, 500), (5, 429)]


## Challenge 5 — Driver events
What is observed vs inferred? Can you find a reliable driver-arrival-at-restaurant event?

In [10]:
with open(BASE/'data'/'driver_events.json') as f: driver_events=json.load(f)
driver_events[0]

{'driver_id': 'D001',
 'events': [{'order_id': 'O00162',
   'type': 'assigned',
   'timestamp': '2026-08-26T17:53:25.977803'},
  {'order_id': 'O00162',
   'type': 'gps_ping',
   'timestamp': '2026-08-26T18:10:59.372420',
   'lat': 12.826339,
   'lon': 77.448282},
  {'order_id': 'O00162',
   'type': 'gps_ping',
   'timestamp': '2026-08-26T18:28:32.767037',
   'lat': 12.876013,
   'lon': 77.477707},
  {'order_id': 'O00162',
   'type': 'gps_ping',
   'timestamp': '2026-08-26T18:46:06.161653',
   'lat': 12.925686,
   'lon': 77.507133},
  {'order_id': 'O00162',
   'type': 'gps_ping',
   'timestamp': '2026-08-26T19:03:39.556270',
   'lat': 12.97536,
   'lon': 77.536558},
  {'order_id': 'O00162',
   'type': 'picked_up',
   'timestamp': '2026-08-26T18:27:32.131791'},
  {'order_id': 'O00162',
   'type': 'delivered',
   'timestamp': '2026-08-26T19:21:12.950887'},
  {'order_id': 'O00180',
   'type': 'assigned',
   'timestamp': '2026-08-08T18:05:08.439625'},
  {'order_id': 'O00180',
   'type': 'gp

In [11]:
flat = pd.DataFrame([{"driver_id": d["driver_id"], **e} for d in driver_events for e in d["events"]])
print(flat["type"].value_counts().to_dict())
print("any explicit restaurant-arrival event?", any("arriv" in t for t in flat["type"].unique()))
pings = flat[flat["type"] == "gps_ping"].assign(ts=lambda x: pd.to_datetime(x["timestamp"], format="mixed")).sort_values(["order_id", "ts"])
print("median minutes between GPS pings:", round((pings.groupby("order_id")["ts"].diff().dt.total_seconds() / 60).median(), 1))

{'gps_ping': 5371, 'assigned': 1600, 'picked_up': 1532, 'delivered': 1532}
any explicit restaurant-arrival event? False
median minutes between GPS pings: 13.4


**Answer:**
- **Observed** (stored events): `assigned`, `picked_up` and `delivered` driver-app taps.
- **Inferred only:** the driver's arrival at the restaurant. **No explicit arrival event exists.** The only possible signal is GPS pings, which come about **every 13 minutes**, too coarse to time an arrival.
- **Consequence:** the order-to-pickup delay cannot be split into *restaurant not ready* vs *driver not there yet*.

## Final FDE recommendation
Prepare one slide: problem size, evidence, uncertainty, missing instrumentation, next step.

**Would you build the AI delay predictor now? Why or why not?**

### Final recommendation

**No, we would not build the AI delay predictor now.**
- **Problem size:** 56.4% of delivered orders are late (843 / 1,495), but the median miss is only about 8 minutes.
- **Evidence:** delay accumulates mainly **before pickup**. Traffic and rain are associated with it, but the ETA already stretches for them.
- **Uncertainty:** we cannot tell whether the pickup delay is the restaurant or the driver, because there is no arrival event. 37 delivered orders have no delivery time. The "late" definition is not agreed.
- **Missing instrumentation:** `driver_arrived_at_restaurant` (geofence) and `food_ready` (restaurant tablet) events; timestamped ETA revisions.
- **Next step:** capture those two events for 2–4 weeks and agree the KPI definition with its owner. Then decide whether the fix is operational (restaurant readiness, dispatch timing) or predictive.

In [12]:
con.close()
if api_proc is not None:
    api_proc.terminate()